# Corte 3, Punto 3: resiliencia y resultados idempotentes

Por: Gabriel Rodriguez y Jonatan Palomares

**Subtareas:** T3.1 a T3.5 (checkpoint en MinIO, destino idempotente, deduplicación en el flujo y pruebas de fallo).

Entrada: el flujo del notebook 05.

_Por hacer._

## Configuración

Las rutas están en `src/pamd/config.py` y la sesión de Spark con MinIO en `src/pamd/spark_session.py`. Las credenciales vienen de las variables de entorno definidas en `.env`.

In [1]:
# Importación de librerías

import os
import time
from time import perf_counter
from statistics import median
import random
import math
import json
import urllib.request
from IPython.display import display, Markdown

import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.types import DateType, TimestampType
from pyspark import StorageLevel

from pamd.config import (
    INPUT_CSV,
    BRONZE_PARQUET_UNPARTITIONED,
    BRONZE_PARQUET,
    QUARANTINE_PARQUET,
    STREAMING_LOTES,
    STREAMING_ENTRADA,
    STREAMING_TARDIOS,
    STREAMING_CHECKPOINT,
    WRITE_MODE,
    OUTPUT_PARTITIONS,
)
from pamd.spark_session import get_spark
from pamd.flujo import (
    entregar_lote, vaciar, crear_carpeta, leer_flujo, con_mes_y_departamento, ventanas_mensuales, TablaEnMemoria,
    archivos_por_disparo, lotes_preparados, contenido_checkpoint, leer_texto, copiar_carpeta,
)

# Spark conectado a MinIO; las credenciales se leen del archivo .env
spark = get_spark()


## T3.1 Punto de control (checkpoint) en MinIO

El flujo del notebook 05 guarda su punto de control en `streaming/checkpoint/ventanas` (`STREAMING_CHECKPOINT` en `src/pamd/config.py`), una ruta de MinIO que solo usa este flujo. La prueba repite la simulación de T2.3 con los mismos lotes y la misma consulta (`src/pamd/flujo.py`), pero **detiene la consulta después del lote 11 y la vuelve a encender con el mismo checkpoint**. Si el checkpoint funciona, al reiniciar no vuelve a leer los 11 lotes ya confirmados y sigue con el 12.

In [2]:
# T3.1 · Parámetros: los mismos del flujo de T2.1 (notebook 05); si se cambian allí, se cambian aquí
TOLERANCIA = "75 days"
SIN_DEPARTAMENTO = "SIN DEPARTAMENTO"
PARTICIONES_ESTADO = 4
DISPARO = "2 seconds"
PAUSA_SEGUNDOS = 5
DETENER_TRAS_LOTE = 11  # la consulta se detiene después del lote 11 (el tardío dentro de la tolerancia)

spark.conf.set("spark.sql.shuffle.partitions", PARTICIONES_ESTADO)
orden = lotes_preparados(spark, STREAMING_LOTES)
filas_lote = {r["archivo"]: r["count"] for r in spark.read.parquet(STREAMING_LOTES).groupBy("archivo").count().collect()}
assert len(orden) == len(filas_lote) == 22, "Faltan lotes de T2.2 en MinIO"

tabla = TablaEnMemoria()  # el mismo destino antes y después del reinicio

def encender():
    """La consulta de T2.3 a T2.5, siempre con el mismo checkpoint."""
    return (
        ventanas_mensuales(leer_flujo(spark, STREAMING_ENTRADA), TOLERANCIA, SIN_DEPARTAMENTO)
        .writeStream
        .queryName("ventanas_mensuales")
        .outputMode("update")
        .foreachBatch(tabla.guardar)
        .option("checkpointLocation", STREAMING_CHECKPOINT)
        .trigger(processingTime=DISPARO)
        .start()
    )

def entregar(consulta, archivos):
    for i, archivo in enumerate(archivos):
        if i:
            time.sleep(PAUSA_SEGUNDOS)
        entregar_lote(spark, STREAMING_LOTES, STREAMING_ENTRADA, archivo)
        consulta.processAllAvailable()
        print(f"{archivo}: entregado y procesado")

def registro(consulta):
    """Un renglón por disparo (el último progreso publicado de cada uno)."""
    por_disparo = {p["batchId"]: p for p in consulta.recentProgress}
    return [
        {"disparo": b, "filas leídas": p["numInputRows"], "marca de agua": p["eventTime"].get("watermark", "")[:10],
         "ventanas en estado": sum(s["numRowsTotal"] for s in p["stateOperators"])}
        for b, p in sorted(por_disparo.items())
    ]

print(f"{len(orden)} lotes preparados; la consulta se detiene tras {orden[DETENER_TRAS_LOTE - 1]}")

22 lotes preparados; la consulta se detiene tras lote_11_tardio_dentro


In [3]:
# T3.1 · Antes del reinicio: simulación limpia hasta el lote 11 y parada de la consulta
vaciar(spark, STREAMING_ENTRADA, STREAMING_CHECKPOINT, STREAMING_TARDIOS)
crear_carpeta(spark, STREAMING_ENTRADA)

consulta = encender()
entregar(consulta, orden[:DETENER_TRAS_LOTE])
time.sleep(1)
registro_antes = registro(consulta)
ultimo_antes = consulta.lastProgress["batchId"]
consulta.stop()

display(pd.DataFrame(registro_antes))
print(f"Consulta detenida. Último disparo confirmado: {ultimo_antes}")

lote_01_regular: entregado y procesado
lote_02_regular: entregado y procesado
lote_03_regular: entregado y procesado
lote_04_regular: entregado y procesado
lote_05_regular: entregado y procesado
lote_06_regular: entregado y procesado
lote_07_regular: entregado y procesado
lote_08_regular: entregado y procesado
lote_09_regular: entregado y procesado
lote_10_regular: entregado y procesado
lote_11_tardio_dentro: entregado y procesado


,disparo,filas leídas,marca de agua,ventanas en estado
0,0,27889,1970-01-01,381
1,1,0,2018-01-16,96
2,2,25620,2018-01-16,388
3,3,0,2018-10-18,96
4,4,28617,2018-10-18,433
5,5,0,2019-08-18,102
6,6,27177,2019-08-18,433
7,7,0,2020-06-18,100
8,8,23720,2020-06-18,332
9,9,0,2021-01-16,101


Consulta detenida. Último disparo confirmado: 20


### Qué guarda el checkpoint

In [4]:
# T3.1 · Contenido del checkpoint después de la parada
archivos_ck = contenido_checkpoint(spark, STREAMING_CHECKPOINT)
QUE_GUARDA = {
    "metadata": "identificador de la consulta",
    "offsets": "un archivo por disparo, escrito ANTES de procesarlo: hasta qué archivo de la fuente llega y la marca de agua",
    "commits": "un archivo por disparo, escrito DESPUÉS de procesarlo: el disparo quedó confirmado",
    "sources": "los archivos de la entrada que la fuente ya encontró, en orden",
    "state": "el estado de la agregación (acumulado de cada ventana abierta), por partición",
}
carpeta = lambda ruta: ruta.split("/")[0]
contenido = pd.DataFrame([
    {"carpeta": c, "archivos": sum(1 for r, _ in archivos_ck if carpeta(r) == c),
     "bytes": sum(t for r, t in archivos_ck if carpeta(r) == c), "qué guarda": QUE_GUARDA.get(c, "")}
    for c in sorted({carpeta(r) for r, _ in archivos_ck})
])
display(contenido)

# Archivos ya leídos, según sources/0 y offsets/
leidos = archivos_por_disparo(spark, STREAMING_CHECKPOINT)
ya_leidos = list(dict.fromkeys(leidos[b] for b in sorted(leidos)))
print(f"Archivos ya leídos ({len(ya_leidos)}):", ", ".join(ya_leidos))

# El último disparo: lo que se planeó (offsets) y su confirmación (commits)
print(f"offsets/{ultimo_antes}:", *leer_texto(spark, f"{STREAMING_CHECKPOINT}/offsets/{ultimo_antes}")[1:], sep="\n  ")
print(f"commits/{ultimo_antes}:", *leer_texto(spark, f"{STREAMING_CHECKPOINT}/commits/{ultimo_antes}")[1:], sep="\n  ")
print(f"Estado: {sum(1 for r, _ in archivos_ck if r.startswith('state/'))} archivos en {PARTICIONES_ESTADO} particiones; "
      f"{registro_antes[-1]['ventanas en estado']} ventanas abiertas al detener la consulta")
assert ya_leidos == orden[:DETENER_TRAS_LOTE], "El checkpoint no registra exactamente los lotes entregados"
assert {"metadata", "offsets", "commits", "sources", "state"} <= set(contenido["carpeta"])

,carpeta,archivos,bytes,qué guarda
0,commits,21,861,"un archivo por disparo, escrito DESPUÉS de pro..."
1,metadata,1,45,identificador de la consulta
2,offsets,21,13975,"un archivo por disparo, escrito ANTES de proce..."
3,sources,11,2129,los archivos de la entrada que la fuente ya en...
4,state,89,145750,el estado de la agregación (acumulado de cada ...


Archivos ya leídos (11): lote_01_regular, lote_02_regular, lote_03_regular, lote_04_regular, lote_05_regular, lote_06_regular, lote_07_regular, lote_08_regular, lote_09_regular, lote_10_regular, lote_11_tardio_dentro
offsets/20:
  {"batchWatermarkMs":1676419200000,"batchTimestampMs":1794791838112,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.join.stateFormatVersion":"2","spark.sql.streaming.stateStore.compression.codec":"lz4","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.shuffle.partitions":"4"}}
  {"logOffset":10}
commits/20:
  {"nextBatchWatermarkMs":1676419200000}
Estado: 89 archivos en 4 particiones; 101 ventana

### Reinicio con el mismo checkpoint

In [5]:
# T3.1 · Después del reinicio: misma consulta, mismo checkpoint
consulta = encender()
consulta.processAllAvailable()  # todavía no ha llegado ningún lote nuevo
time.sleep(1)
leidas_sin_lotes_nuevos = sum(p["numInputRows"] for p in consulta.recentProgress)
print(f"Al reiniciar, sin lotes nuevos, la consulta leyó {leidas_sin_lotes_nuevos} filas")

entregar(consulta, orden[DETENER_TRAS_LOTE:])
time.sleep(1)
registro_despues = registro(consulta)
consulta.stop()
display(pd.DataFrame(registro_despues))

Al reiniciar, sin lotes nuevos, la consulta leyó 0 filas
lote_12_regular: entregado y procesado
lote_13_regular: entregado y procesado
lote_14_regular: entregado y procesado
lote_15_regular: entregado y procesado
lote_16_regular: entregado y procesado
lote_17_tardio_fuera: entregado y procesado
lote_18_regular: entregado y procesado
lote_19_regular: entregado y procesado
lote_20_regular: entregado y procesado
lote_21_regular: entregado y procesado
lote_22_regular: entregado y procesado


,disparo,filas leídas,marca de agua,ventanas en estado
0,21,26337,2023-02-15,236
1,22,0,2023-06-18,102
2,23,23285,2023-06-18,204
3,24,0,2023-09-17,102
4,25,28015,2023-09-17,303
5,26,0,2024-03-18,102
6,27,26354,2024-03-18,238
7,28,0,2024-07-18,102
8,29,25064,2024-07-18,236
9,30,0,2024-11-18,100


### Criterio de aceptación

In [6]:
# T3.1 · Tras detener y reiniciar con el mismo checkpoint, no se releen los lotes confirmados y se sigue con el siguiente
leidos = archivos_por_disparo(spark, STREAMING_CHECKPOINT)
con_datos_antes = [r["disparo"] for r in registro_antes if r["filas leídas"] > 0]
con_datos_despues = [r["disparo"] for r in registro_despues if r["filas leídas"] > 0]
primer_disparo = registro_despues[0]["disparo"]

display(pd.DataFrame([
    {"momento": "antes del reinicio" if b in con_datos_antes else "después del reinicio", "disparo": b,
     "archivo": leidos[b], "filas leídas": r["filas leídas"], "filas del lote": filas_lote[leidos[b]]}
    for b, r in sorted({**{r["disparo"]: r for r in registro_antes}, **{r["disparo"]: r for r in registro_despues}}.items())
    if r["filas leídas"] > 0
]))

assert leidas_sin_lotes_nuevos == 0, "Al reiniciar se volvieron a leer lotes ya confirmados"
assert primer_disparo == ultimo_antes + 1, "La numeración de disparos no continúa después del reinicio"
assert leidos[con_datos_despues[0]] == orden[DETENER_TRAS_LOTE], "Después del reinicio no siguió con el lote siguiente"
assert [leidos[b] for b in con_datos_antes + con_datos_despues] == orden, "Algún lote se leyó dos veces o se saltó"
assert sum(r["filas leídas"] for r in registro_antes + registro_despues) == sum(filas_lote.values()), "Las filas leídas no suman los lotes"
print(f"Antes del reinicio: disparos {con_datos_antes[0]} a {ultimo_antes} ({DETENER_TRAS_LOTE} lotes). "
      f"Después: empieza en el disparo {primer_disparo} sin releer nada y el primer lote que lee es {orden[DETENER_TRAS_LOTE]}.")
print("Criterio de aceptación de T3.1: OK")

,momento,disparo,archivo,filas leídas,filas del lote
0,antes del reinicio,0,lote_01_regular,27889,27889
1,antes del reinicio,2,lote_02_regular,25620,25620
2,antes del reinicio,4,lote_03_regular,28617,28617
3,antes del reinicio,6,lote_04_regular,27177,27177
4,antes del reinicio,8,lote_05_regular,23720,23720
5,antes del reinicio,10,lote_06_regular,30431,30431
6,antes del reinicio,12,lote_07_regular,26812,26812
7,antes del reinicio,14,lote_08_regular,21842,21842
8,antes del reinicio,16,lote_09_regular,34509,34509
9,antes del reinicio,18,lote_10_regular,20887,20887


Antes del reinicio: disparos 0 a 20 (11 lotes). Después: empieza en el disparo 21 sin releer nada y el primer lote que lee es lote_12_regular.
Criterio de aceptación de T3.1: OK


### Cambiar la consulta invalida el checkpoint

El estado guardado tiene la forma de la consulta que lo creó: la clave (`mes`, `departamento`) y los acumulados de cada medida. Si la consulta cambia (otra clave, otras medidas u otra columna de marca de agua), Spark no puede reutilizar ese estado. La prueba se hace sobre una **copia** del checkpoint en `streaming/prueba_cambio/`, que se borra al terminar; el checkpoint real no se toca.

In [7]:
# T3.1 · Prueba: la misma fuente agrupada solo por mes (sin departamento) sobre una copia del checkpoint
PRUEBA = STREAMING_CHECKPOINT.split("/checkpoint")[0] + "/prueba_cambio"
vaciar(spark, PRUEBA)
copiar_carpeta(spark, STREAMING_CHECKPOINT, f"{PRUEBA}/checkpoint")
crear_carpeta(spark, f"{PRUEBA}/entrada")
entregar_lote(spark, STREAMING_LOTES, f"{PRUEBA}/entrada", orden[-1])

consulta_cambiada = (
    con_mes_y_departamento(leer_flujo(spark, f"{PRUEBA}/entrada"), SIN_DEPARTAMENTO)
    .withWatermark("mes", TOLERANCIA)
    .groupBy("mes")
    .agg(F.count("*").alias("contratos"))
    .writeStream.outputMode("update")
    .foreachBatch(lambda df, _: df.collect())
    .option("checkpointLocation", f"{PRUEBA}/checkpoint")
    .start()
)
try:
    consulta_cambiada.processAllAvailable()
    error = ""
except Exception as e:
    error = str(e)
finally:
    consulta_cambiada.stop()
    vaciar(spark, PRUEBA)

assert "StateSchemaNotCompatible" in error, "Se esperaba que la consulta cambiada no pudiera usar el checkpoint"
lineas_clave = list(dict.fromkeys(l.strip() for l in error.splitlines() if "key schema" in l))
print(*lineas_clave[:2], sep="\n")
print("La consulta cambiada se detuvo con StateSchemaNotCompatible: el estado guardado no tiene su forma.")

- Provided key schema: StructType(StructField(mes,TimestampType,true))
- Existing key schema: StructType(StructField(mes,TimestampType,true),StructField(departamento,StringType,true))
La consulta cambiada se detuvo con StateSchemaNotCompatible: el estado guardado no tiene su forma.


**Qué queda documentado para T3.5.**
- El checkpoint guarda qué archivos ya se leyeron (`sources`), hasta dónde llegó cada disparo y con qué marca de agua (`offsets`), qué disparos quedaron confirmados (`commits`) y el acumulado de las ventanas abiertas (`state`). Al reiniciar, Spark retoma desde el último disparo confirmado: si se cayó después de escribir `offsets` y antes de `commits`, repite ese disparo con exactamente los mismos archivos.
- Cambiar la consulta puede invalidar el checkpoint: otra clave o medidas distintas fallan con `StateSchemaNotCompatible` (prueba de arriba). Otros parámetros quedan fijados en el checkpoint y se ignoran si se cambian después, como `spark.sql.shuffle.partitions` (aparece en `offsets`).
- Borrar el checkpoint hace que el flujo vuelva a leer todos los archivos de la entrada desde cero.
- El checkpoint protege lo que Spark lee y su estado, no el destino: la tabla en memoria de T2 vive en el kernel. Si se reinicia el kernel, Spark retoma bien, pero las ventanas que ya se habían emitido no están en el destino. Por eso el destino idempotente en MinIO de T3.2 es la otra mitad del mecanismo.